<a href="https://colab.research.google.com/github/garam827/LLM_Study/blob/main/%EB%9E%AD%EC%B2%B4%EC%9D%B8_%EC%97%90%EC%9D%B4%EC%A0%84%ED%8A%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ! pip install langchain
# ! pip install langchain-openai

In [6]:
import os
from google.colab import userdata

API_KEY = userdata.get('OPEN_ROUTER')
os.environ['OPENAI_API_KEY'] = API_KEY

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

llm = ChatOpenAI(
	model_name = 'nvidia/nemotron-3-super-120b-a12b:free',
	openai_api_base = 'https://openrouter.ai/api/v1'
)

In [9]:
llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='봤어요, 잘 지냈어요! 요새 잘 지냈는데, 어떻게 지내셨는지 궁금해요. 😊  \n무엇을 말씀드릴까요? 오늘은 기분 좋게 지내고 계신가요? 혹시 나누고 싶은 이야기나 궁금한 점이 있다면 편하게 말씀해 주세요! 🌸', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 21, 'total_tokens': 112, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 23, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1775394815-STtZq6Fk8WIGpow4BfsT', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d5dc6-feb9-7693-a8a9-b67697d2ecb5-0', tool_calls=[], invalid_tool

## @tool 데코레이터

In [11]:
# 랭체인에 시간을 파악하는 도구 추가

from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [12]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [13]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 380, 'total_tokens': 469, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 51, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1775395289-PlCPL31tZn1gIFyXRGR3', 'finish_reason': 'tool_calls', 'logprobs': None},

In [14]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-04-05 22:22:44 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 380, 'total_tokens': 469, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 51, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1775395289-PlCPL31tZn1gIFyXRGR3', 'finish_reason': 'tool_calls', 'logprobs': None

In [15]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 **2026년 4월 5일 22시 22분 44초**입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 469, 'total_tokens': 578, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 65, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1775395393-KJa6ufWwDlX6uDlqRSWy', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d5dcf-cfd6-70c3-b3d2-31ec3c9a0dbc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 469, 'output_tokens': 109, 'total_

## 파이단틱 활용

In [16]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title = "주식 코드", description = "주식 코드 (예: AAPL)")
    period: str = Field(..., title = "기간", description = "주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [17]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown()

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [18]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 641, 'total_tokens': 752, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 69, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1775395632-qBNizFbtxKLVPu8lRMAu', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019d5dd3-74ec-7050-8395-9b180f35b76d-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'chatcmpl-tool-b4900c98341739

In [19]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-03-03 00:00:00-05:00 | 395.09 | 396.34 | 385.39 |  392.43 | 6.26173e+07 |           0 |              0 |\n| 2026-03-04 00:00:00-05:00 | 397.85 | 408.33 | 394.58 |  405.94 | 6.83055e+07 |           0 |              0 |\n| 2026-03-05 00:00:00-05:00 | 401.57 | 408.62 | 399.42 |  405.55 | 5.19259e+07 |           0 |              0 |\n| 2026-03-06 00:00:00-05:00 | 398.09 | 402.35 | 394.21 |  396.73 | 6.40546e+07 |           0 |              0 |\n| 2026-03-09 00:00:00-04:00 | 390.05 | 401.59 | 381.4  |  398.68 | 6.70189e+07 |           0 |              0 |\n| 2026-03-10 00:00:00-04:00 | 402.22 | 406.59 | 398.19 |  399.24 | 5.92587e+07 |           0 |              0 |\n| 2026-03-11 00:00:00-04:0

In [20]:
llm_with_tools.invoke(messages)

AIMessage(content='테슬라(TSLA)의 주가는 약 한 달 전(2026년 3월\u202f5일 종가\u202f≈\u202f405.55달러)와 비교했을 때 현재(2026년 4월\u202f2일 종가\u202f≈\u202f360.59달러)로 **하락**했습니다.\n\n- **변화 폭**: 약\u202f‑44.96달러  \n- **변화 비율**: 약\u202f‑11.1\u202f% (주가가 약 11\u202f% 낮아졌습니다)\n\n따라서, 한 달 전보다 테슬라 주가가 내린 상황입니다. (참고: 최신 데이터는 4월\u202f2일까지이며, 4월\u202f5일 당일 데이터는 아직 반영되지 않았습니다.)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 412, 'prompt_tokens': 2638, 'total_tokens': 3050, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 152, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230